# SmolVLA on a T4: memory check for Run C's freezing (about 15 min of GPU)

Run C trains the vision encoder, the connector and the action expert, and keeps the
language layers of the VLM frozen (gradients still flow back through them to reach the
vision encoder). This notebook measures, before any Run C code is written, the largest
batch that fits on one Kaggle T4 with that freezing: 30 steps each at batch 4, 6 and 8,
mixed precision, float32 master weights, the same settings as Run A otherwise.

It is Run A's notebook (`smolvla_tri_sort_runA_frozen_20k.ipynb`) in `benchmark` mode with
four changes:

* `FREEZE = "runC"`: LeRobot builds the policy with `freeze_vision_encoder=false` and
  `train_expert_only=false`, then the launcher sets `requires_grad=False` on every VLM
  parameter outside `vision_model` and `connector` (the language layers and `lm_head`).
  The launcher prints the trainable parameter count by group; it must read about 198 M
  (vision ~86 M, connector ~12 M, expert and projections ~100 M).
* Batch sizes 4 / 6 / 8, mixed precision only, 30 steps (the first 10 left out of the
  averages). It stops at the first batch that runs out of memory.
* `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, as in the 36k run.
* The table adds `mem_gb`, PyTorch's peak allocated memory in GiB (what LeRobot logs), next
  to the nvidia-smi peak in MB.

What is not in it: gradient accumulation (it does not change the peak: the gradients exist
at every step anyway), the separate vision learning rate (no memory effect) and the
auxiliary position head (a small MLP, well under 1 M parameters).

History. The first version (19/10, batches 16 / 20 / 24) ran out of memory at batch 16
(nvidia-smi peak 14 903 MB). Its estimate wrongly took the 36k run as training all 450 M
parameters. The 36k run already had Run C's freezing (198.1 M trainable, language frozen) and
peaked at 9.21 GiB at batch 4: about 4.7 GiB fixed + 1.1 GiB per sample, so batch 6 about
11.5 GiB and batch 8 about 13.7 GiB (borderline on the T4's ~14.5 GiB). Batch 4 is in the
list to check that this torch build (2.11, CUDA 13) still gives the 36k run's 9.21 GiB.

## Kaggle, step by step

1. *Code* -> *New Notebook* -> *File* -> *Import Notebook* -> this file.
2. *Session options*: *Accelerator* **GPU T4 x2** (one is used), *Internet* **On**,
   *Add Input* -> *Your Work* -> *Datasets* -> `mycobot-tri-sort-160` (only this one).
3. *Run All* in the interactive session (no commit needed). The first driver line must read
   *160 episodes, 28359 frames*; the `[fp32 launcher]` line must show about 198 M trainable.
4. Copy the final table (or download `benchmark.json` from `/kaggle/working`), then stop the
   session so it does not use more GPU quota.


In [ ]:
# ---- 1. Configuration ---------------------------------------------------------
MODE = "benchmark"           # memory check only

TRAIN_STEPS = 20000          # set from the benchmark (see "Choosing TRAIN_STEPS")
TRAIN_BATCH_SIZE = 32        # Kaggle T4 benchmark: 11.2 samples/s, 12.4 GB
TRAIN_AMP = True             # mixed precision (fp16 on a T4); set from the benchmark
FP32_WEIGHTS = True          # cast smolvla_base (bfloat16) to float32; needed for TRAIN_AMP on a T4
SAVE_FREQ = 500              # divides KEEP_EVERY; a session cut loses at most ~23 min
KEEP_EVERY = 5000            # keep the weights of every KEEP_EVERY-th step (1.8 GB each), 0 = newest only
SESSION_HOURS = 10.5         # stop cleanly before Kaggle's 12 h limit (install and output saving included)

BENCH_BATCH_SIZES = [4, 6, 8]
BENCH_AMP = [True]          # Run C trains in mixed precision, as Run A
BENCH_STEPS = 30
BENCH_WARMUP_STEPS = 10      # excluded from the averages
FREEZE = "runC"              # "runC": vision + connector + expert trained, language layers frozen; None = LeRobot default (expert only)

TORCH_BACKEND = "auto"       # uv picks the CUDA build for the driver; e.g. "cu126" to force
VIDEO_BACKEND = None         # None = LeRobot's choice (torchcodec, else pyav); "pyav" to force
NUM_WORKERS = 4
SEED = 1000

import os
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # as in the 36k run

if Path("/kaggle").is_dir():
    PLATFORM, INPUT_ROOT, WORKING = "kaggle", "/kaggle/input", "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PLATFORM = "colab"
    INPUT_ROOT = "/content/drive/MyDrive/smolvla_tri_sort"
    WORKING = INPUT_ROOT + "/working"
LOCAL = "/tmp/smolvla"          # venv, model cache, dataset copy: not saved as output
VENV = LOCAL + "/venv"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(PLATFORM, INPUT_ROOT, WORKING)

In [ ]:
# ---- 2. Python 3.12 environment with LeRobot 0.6.1 (about 3-5 min) ------------
import subprocess, sys
from pathlib import Path

CONSTRAINTS = """aiohappyeyeballs==2.7.1
aiohttp==3.14.4
aiosignal==1.4.0
anyio==4.15.1
av==15.1.0
certifi==2026.7.22
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
cmake==4.1.3
datasets==4.8.5
dill==0.4.1
draccus==0.11.6
einops==0.8.2
Farama-Notifications==0.0.6
frozenlist==1.8.0
fsspec==2026.2.0
grpcio==1.84.0
gymnasium==1.4.0
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.33.0
idna==3.20
jsonlines==4.0.0
lerobot==0.6.1
mergedeep==1.3.4
mpmath==1.3.0
multidict==6.9.1
multiprocess==0.70.19
mypy_extensions==1.1.0
networkx==3.6.1
numpy==2.2.6
opencv-python-headless==4.13.0.92
packaging==25.0
pandas==2.3.3
propcache==0.5.4
protobuf==7.36.2
pyarrow==26.0.0
requests==2.34.2
safetensors==0.8.0
scipy==1.18.1
setuptools==78.1.0
sympy==1.14.0
termcolor==3.3.0
toml==0.10.2
tqdm==4.70.1
typing-inspect==0.9.0
typing_extensions==4.16.0
tzdata==2026.5
urllib3==2.8.0
xxhash==4.0.1
yarl==1.25.1
accelerate==1.15.0
transformers==5.5.4
tokenizers==0.22.2
num2words==0.5.14
"""

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not Path(VENV, "bin", "lerobot-train").exists():
    Path(LOCAL).mkdir(parents=True, exist_ok=True)
    Path(LOCAL, "constraints.txt").write_text(CONSTRAINTS)
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --python 3.12 {VENV}")
    sh(f"{sys.executable} -m uv pip install --python {VENV}/bin/python --torch-backend={TORCH_BACKEND} "
       f"-c {LOCAL}/constraints.txt 'lerobot[smolvla,training]==0.6.1'")
sh(f"{VENV}/bin/python -c \"import torch, lerobot; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
   f"'|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU', '| lerobot', lerobot.__version__)\"")
Path(WORKING).mkdir(parents=True, exist_ok=True)
sh(f"{sys.executable} -m uv pip freeze --python {VENV}/bin/python > {WORKING}/pip_freeze.txt")
sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")

## 3. Driver
The same file as `training/smolvla_kaggle/smolvla_kaggle_driver.py` in the repository, tested there on CPU (benchmark, stop at the time limit, resume from an attached output).

In [ ]:
%%writefile /tmp/smolvla_kaggle_driver.py
#!/usr/bin/env python3
"""SmolVLA fine-tune of local/mycobot_tri_sort on a Kaggle (or Colab) GPU.

Two modes, read from the JSON config written by the notebook's first cell:
  benchmark  short runs over batch sizes and precisions; prints s/step,
             samples/s, data-loading time, peak GPU memory and projections.
  train      fresh run, or resumes automatically from the newest complete
             checkpoint found under the input root (a previous run's output
             attached as input).

Standard library only: lerobot-train is run from the venv as a subprocess.
Paths come from the config so the same file runs on Kaggle, Colab and in the
gazebo_to_lerobot container (CPU) where it was tested.
"""
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO_ID = 'local/mycobot_tri_sort'
CAMERA_RENAME = {
    'observation.images.top': 'observation.images.camera1',
    'observation.images.right': 'observation.images.camera2',
    'observation.images.left': 'observation.images.camera3',
}
# smolvla_base loads in bfloat16 (447 M of 450 M parameters, 97 M of the 100 M trained).
# A T4 has no bfloat16 tensor cores, and fp16 mixed precision cannot unscale bfloat16
# gradients ("_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for
# 'BFloat16', Kaggle 2026-10-12). Casting the policy to float32 right after LeRobot builds
# it keeps float32 master weights; with use_amp the matmuls then run in float16.
FP32_LAUNCHER = '''import os
import sys
from collections import Counter
import lerobot.scripts.lerobot_train as lt

_make_policy = lt.make_policy


FREEZE = os.environ.get("SMOLVLA_FREEZE") or None


def group(name):
    if "vlm_with_expert.vlm." not in name:
        return "expert and projections"
    if "vision_model" in name:
        return "vision"
    if "connector" in name:
        return "connector"
    return "language"


def make_policy_fp32(*args, **kwargs):
    policy = _make_policy(*args, **kwargs).float()
    if FREEZE == "runC":
        # Language layers and lm_head frozen; gradients still flow through them to the vision encoder.
        for name, p in policy.named_parameters():
            if group(name) == "language":
                p.requires_grad = False
    trained = Counter(str(p.dtype) for p in policy.parameters() if p.requires_grad)
    by_group = Counter()
    for name, p in policy.named_parameters():
        if p.requires_grad:
            by_group[group(name)] += p.numel()
    print(f"[fp32 launcher] policy cast to float32; freeze={FREEZE}; trainable tensors by dtype: {dict(trained)}; "
          f"trainable parameters by group (M): { {k: round(v / 1e6, 1) for k, v in sorted(by_group.items())} }, "
          f"total {sum(by_group.values()) / 1e6:.1f} M", flush=True)
    return policy


# DataLoader workers start with "spawn" and re-import this file as __mp_main__.
if __name__ == "__main__":
    lt.make_policy = make_policy_fp32
    sys.exit(lt.main())
'''
# lerobot prints the step through format_big_number ("step:950", then "step:1K"), so the exact
# count comes from the tqdm prefix on the same line ("1000/20000 [").
STEP_RE = re.compile(r'step:([\d.]+)([KMB]?)\s.*?loss:([\d.eE+-]+).*?updt_s:([\d.]+).*?data_s:([\d.]+)')
TQDM_RE = re.compile(r'(\d+)/\d+ \[')
MEM_RE = re.compile(r'mem_gb:([\d.]+)')
SUFFIX = {'': 1, 'K': 1_000, 'M': 1_000_000, 'B': 1_000_000_000}
OOM_MARKERS = ('OutOfMemoryError', 'CUDA out of memory')


def log(msg):
    print(f'[driver {time.strftime("%H:%M:%S")}] {msg}', flush=True)


def find_dataset(input_root, scratch):
    for root_dir in (Path(input_root), Path(scratch) / 'unzipped'):
        for info in sorted(root_dir.rglob('meta/info.json')):
            root = info.parent.parent
            features = json.loads(info.read_text()).get('features', {})
            if all(k in features for k in CAMERA_RENAME) and (root / 'videos').is_dir():
                return root
        archives = sorted(Path(input_root).rglob('mycobot_tri_sort.zip'))
        if archives and root_dir == Path(input_root):
            log(f'extracting {archives[0]}')
            shutil.unpack_archive(archives[0], Path(scratch) / 'unzipped')
    sys.exit(f'No mycobot_tri_sort dataset (meta/info.json with top/right/left, or mycobot_tri_sort.zip) '
             f'under {input_root}. Attach the uploaded dataset as an input.')


def stage_dataset(cfg):
    src = find_dataset(cfg['input_root'], cfg['scratch'])
    dst = Path(cfg['data_root']) / REPO_ID
    if not (dst / 'meta' / 'info.json').exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns('images'))
    info = json.loads((dst / 'meta' / 'info.json').read_text())
    log(f'dataset {src} -> {dst}: {info["total_episodes"]} episodes, {info["total_frames"]} frames, '
        f'fps {info["fps"]}')
    return dst


def drop_incomplete_checkpoints(run_dir):
    """After lerobot-train has exited, delete checkpoints newer than `last`: lerobot moves `last`
    only once a save is complete, so a newer one was cut mid-write and must not be resumed from."""
    last = Path(run_dir) / 'checkpoints' / 'last'
    if not last.is_symlink():
        return
    for d in last.parent.iterdir():
        if d.is_dir() and not d.is_symlink() and d.name.isdigit() and int(d.name) > int(last.resolve().name):
            shutil.rmtree(d, ignore_errors=True)
            log(f'deleted {d.name}: written past `last`, incomplete')


def complete_checkpoints(root):
    found = []
    for cfg_file in Path(root).rglob('checkpoints/*/pretrained_model/train_config.json'):
        ckpt = cfg_file.parent.parent
        if ckpt.is_symlink() or not ckpt.name.isdigit():
            continue
        if (ckpt / 'training_state').is_dir() and any((ckpt / 'pretrained_model').glob('*.safetensors')):
            found.append((int(ckpt.name), ckpt))
    return sorted(found)


def prune_checkpoints(run_dir, keep_every=0, kept_root=None):
    """Delete checkpoints older than the one `last` points to.

    lerobot moves `last` only after a save completes, and a checkpoint newer than `last` may be
    one still being written, so both are kept. Safe to call at any moment. With keep_every, the
    weights of every keep_every-th step move to kept_root/<step>/pretrained_model first (no
    optimizer state: 1.8 GB instead of ~3.5), so earlier checkpoints can be scored in Gazebo.
    """
    ckpt_root = Path(run_dir) / 'checkpoints'
    last = ckpt_root / 'last'
    if not last.is_symlink():
        return
    keep = int(last.resolve().name)
    for d in ckpt_root.iterdir():
        if d.is_dir() and not d.is_symlink() and d.name.isdigit() and int(d.name) < keep:
            if keep_every and int(d.name) % keep_every == 0 and (d / 'pretrained_model').is_dir():
                dst = Path(kept_root) / d.name / 'pretrained_model'
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(d / 'pretrained_model'), str(dst))
                    log(f'kept weights of step {d.name} in {dst}')
            shutil.rmtree(d, ignore_errors=True)
            log(f'pruned {d.name}')


class GpuMemSampler(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.peak_mb = None
        self.stop = threading.Event()

    def run(self):
        if not shutil.which('nvidia-smi'):
            return
        while not self.stop.wait(1.0):
            try:
                out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits',
                                      '-i', os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',')[0]],
                                     capture_output=True, text=True, timeout=10).stdout.strip()
                self.peak_mb = max(self.peak_mb or 0, int(out.splitlines()[0]))
            except (ValueError, IndexError, subprocess.TimeoutExpired):
                pass


def run_lerobot(cfg, args, log_path, deadline=None, prune_dir=None):
    """Stream lerobot-train output; stop it cleanly at the deadline. Returns (rc, steps, oom, peak_mb, stopped)."""
    keep = dict(keep_every=cfg.get('keep_every', 0), kept_root=Path(cfg['working']) / 'kept')
    if cfg.get('fp32_weights', True):
        launcher = Path(cfg['scratch']) / 'lerobot_train_fp32.py'
        launcher.write_text(FP32_LAUNCHER)
        cmd = [str(Path(cfg['venv']) / 'bin' / 'python'), str(launcher)] + args
    else:
        cmd = [str(Path(cfg['venv']) / 'bin' / 'lerobot-train')] + args
    env = dict(os.environ, HF_HOME=cfg['hf_home'], WANDB_MODE='disabled', PYTHONUNBUFFERED='1',
               SMOLVLA_FREEZE=cfg.get('freeze') or '')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log('run: ' + ' '.join(cmd))
    sampler = GpuMemSampler()
    sampler.start()
    steps, oom, stopped = [], False, False
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                            start_new_session=True)
    watchdog = None
    if deadline:
        def stop_at_deadline():
            nonlocal stopped
            while proc.poll() is None:
                if time.time() > deadline:
                    stopped = True
                    log('session time limit reached: stopping training (resume from the last checkpoint)')
                    os.killpg(proc.pid, signal.SIGINT)
                    try:
                        proc.wait(timeout=180)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid, signal.SIGKILL)
                    return
                time.sleep(10)
        watchdog = threading.Thread(target=stop_at_deadline, daemon=True)
        watchdog.start()
    # Pruning at every step line and every "Checkpoint" line keeps at most two checkpoints on disk:
    # the one `last` points to and the one being written.
    with open(log_path, 'a') as f:
        for line in proc.stdout:
            f.write(line)
            if any(m in line for m in OOM_MARKERS):
                oom = True
            m = STEP_RE.search(line)
            if m:
                t = TQDM_RE.search(line[:m.start()])
                step = int(t[1]) if t else int(float(m[1]) * SUFFIX[m[2]])
                mem = MEM_RE.search(line)
                steps.append({'step': step, 'loss': float(m[3]), 'updt_s': float(m[4]), 'data_s': float(m[5]),
                              'mem_gb': float(mem[1]) if mem else None})
            if prune_dir and (m or 'Checkpoint policy after step' in line):
                prune_checkpoints(prune_dir, **keep)
            if m or 'Checkpoint' in line or '[fp32 launcher]' in line or (
                    ('Traceback' in line or 'Error' in line) and 'torchcodec' not in line
                    and 'libnppicc' not in line and 'Could not load this library' not in line):
                print(line.rstrip()[-300:], flush=True)
    rc = proc.wait()
    sampler.stop.set()
    if prune_dir:
        prune_checkpoints(prune_dir, **keep)
    return rc, steps, oom, sampler.peak_mb, stopped


def common_args(cfg, data_dir, batch_size, steps, use_amp):
    return [
        '--policy.path=lerobot/smolvla_base',
        f'--policy.device={cfg["device"]}',
        '--policy.push_to_hub=false',
        f'--policy.use_amp={str(use_amp).lower()}',
        f'--policy.scheduler_decay_steps={steps}',
        f'--policy.scheduler_warmup_steps={min(1000, max(1, steps // 20))}',
        f'--dataset.repo_id={REPO_ID}',
        f'--dataset.root={data_dir}',
        '--rename_map=' + json.dumps(CAMERA_RENAME, separators=(',', ':')),
        f'--batch_size={batch_size}',
        f'--steps={steps}',
        f'--num_workers={cfg["num_workers"]}',
        '--wandb.enable=false',
        f'--seed={cfg["seed"]}',
    ] + ([f'--dataset.video_backend={cfg["video_backend"]}'] if cfg.get('video_backend') else []) + (
        ['--policy.freeze_vision_encoder=false', '--policy.train_expert_only=false']
        if cfg.get('freeze') == 'runC' else [])


def benchmark(cfg, data_dir):
    work = Path(cfg['working'])
    rows = []
    for use_amp in cfg['bench_amp']:
        for bs in cfg['bench_batch_sizes']:
            out = Path(cfg['scratch']) / f'bench_bs{bs}_amp{int(use_amp)}'
            shutil.rmtree(out, ignore_errors=True)
            args = common_args(cfg, data_dir, bs, cfg['bench_steps'], use_amp) + [
                '--log_freq=1', '--save_checkpoint=false', f'--output_dir={out}']
            t0 = time.time()
            rc, steps, oom, peak, _ = run_lerobot(cfg, args, work / 'logs' / f'bench_bs{bs}_amp{int(use_amp)}.log')
            shutil.rmtree(out, ignore_errors=True)
            steady = [s for s in steps if s['step'] > cfg['bench_warmup_steps']]
            row = {'batch_size': bs, 'amp': use_amp, 'rc': rc, 'oom': oom, 'peak_gpu_mb': peak,
                   'wall_s': round(time.time() - t0)}
            if steady and rc == 0:
                updt = sum(s['updt_s'] for s in steady) / len(steady)
                data = sum(s['data_s'] for s in steady) / len(steady)
                row.update(s_per_step=round(updt + data, 3), updt_s=round(updt, 3), data_s=round(data, 3),
                           samples_per_s=round(bs / (updt + data), 2),
                           mem_gb=max((s['mem_gb'] for s in steps if s['mem_gb'] is not None), default=None))
            rows.append(row)
            log(f'benchmark {row}')
            if oom or rc != 0:
                break
    (work / 'benchmark.json').write_text(json.dumps(rows, indent=2))
    ok = [r for r in rows if 's_per_step' in r]
    print('\n batch  amp    s/step  data_s  samples/s  mem_gb  peak GPU MB   h for train_steps   steps in 11 h')
    for r in ok:
        hours = r['s_per_step'] * cfg['train_steps'] / 3600
        print(f" {r['batch_size']:5d}  {str(r['amp']):5s}  {r['s_per_step']:6.2f}  {r['data_s']:6.2f}  "
              f"{r['samples_per_s']:9.2f}  {str(r['mem_gb']):>6s}  {str(r['peak_gpu_mb']):>11s}   {hours:17.1f}   "
              f"{int(11 * 3600 / r['s_per_step']):13d}")
    for r in rows:
        if 's_per_step' not in r:
            print(f" {r['batch_size']:5d}  {str(r['amp']):5s}  {'out of memory' if r['oom'] else 'failed rc=' + str(r['rc'])}"
                  f" (log: logs/bench_bs{r['batch_size']}_amp{int(r['amp'])}.log)")
    if ok:
        best = max(ok, key=lambda r: r['samples_per_s'])
        print(f"\nFastest: batch {best['batch_size']}, amp {best['amp']}: {best['samples_per_s']} samples/s. "
              f"train_steps={cfg['train_steps']} at that batch would take "
              f"{best['s_per_step'] * cfg['train_steps'] / 3600:.1f} h of GPU.")
        if best['data_s'] > 0.3 * best['s_per_step']:
            print('Note: data loading is over 30 % of each step; video decoding on the CPU is limiting.')


def train(cfg, data_dir):
    work = Path(cfg['working'])
    run_dir = work / 'run'
    # A Kaggle session's output holds only what it wrote: carry the kept weights of earlier
    # sessions (attached as input) forward, so the last session's output has them all.
    for kept in sorted(Path(cfg['input_root']).rglob('kept/*/pretrained_model')):
        dst = work / 'kept' / kept.parent.name / 'pretrained_model'
        if kept.parent.name.isdigit() and not dst.exists():
            shutil.copytree(kept, dst)
            log(f'carried kept weights of step {kept.parent.name} forward')
    previous = complete_checkpoints(cfg['input_root'])
    if previous:
        step, ckpt = previous[-1]
        log(f'resuming from step {step}: {ckpt}')
        # A session cut right after a kept step leaves that checkpoint only in the input,
        # where pruning never sees it.
        dst = work / 'kept' / ckpt.name / 'pretrained_model'
        if cfg.get('keep_every') and step % cfg['keep_every'] == 0 and not dst.exists():
            shutil.copytree(ckpt / 'pretrained_model', dst)
            log(f'kept weights of step {ckpt.name} (resumed from) in {dst}')
        args = [f'--config_path={ckpt / "pretrained_model" / "train_config.json"}', '--resume=true']
    else:
        if run_dir.exists() and complete_checkpoints(run_dir):
            sys.exit(f'{run_dir} already holds checkpoints from this session; attach it as input to resume, '
                     'or delete it to start over.')
        shutil.rmtree(run_dir, ignore_errors=True)
        log('no previous checkpoint attached: fresh run from lerobot/smolvla_base')
        args = common_args(cfg, data_dir, cfg['train_batch_size'], cfg['train_steps'], cfg['train_amp']) + [
            f'--save_freq={cfg["save_freq"]}', f'--log_freq={cfg["log_freq"]}', f'--output_dir={run_dir}']
    deadline = time.time() + cfg['session_hours'] * 3600
    rc, steps, oom, peak, stopped = run_lerobot(cfg, args, work / 'logs' / f'train_{time.strftime("%Y%m%d_%H%M%S")}.log',
                                                deadline=deadline, prune_dir=run_dir)
    drop_incomplete_checkpoints(run_dir)
    done = complete_checkpoints(run_dir)
    summary = {'rc': rc, 'oom': oom, 'stopped_at_time_limit': stopped, 'peak_gpu_mb': peak,
               'last_logged_step': steps[-1]['step'] if steps else None,
               'last_logged_loss': steps[-1]['loss'] if steps else None,
               'checkpoint': str(done[-1][1]) if done else None,
               'target_steps': cfg['train_steps']}
    (work / 'train_summary.json').write_text(json.dumps(summary, indent=2))
    log(f'summary {summary}')
    if not done:
        sys.exit('No complete checkpoint was written in this session.')
    if done[-1][0] >= cfg['train_steps']:
        log(f'TRAINING COMPLETE. Download {done[-1][1] / "pretrained_model"}')
    else:
        log(f'Stopped at step {done[-1][0]} of {cfg["train_steps"]}. Next session: attach this output as input '
            'and run again in train mode; it resumes from this checkpoint.')


def main():
    cfg = json.loads(Path(sys.argv[1]).read_text())
    for k in ('working', 'scratch', 'data_root', 'hf_home'):
        Path(cfg[k]).mkdir(parents=True, exist_ok=True)
    data_dir = stage_dataset(cfg)
    if cfg['mode'] == 'benchmark':
        benchmark(cfg, data_dir)
    elif cfg['mode'] == 'train':
        train(cfg, data_dir)
    else:
        sys.exit(f'unknown mode {cfg["mode"]!r}: use "benchmark" or "train"')


if __name__ == '__main__':
    main()

In [ ]:
# ---- 4. Run ------------------------------------------------------------------
import json, subprocess, sys
from pathlib import Path

Path(WORKING).mkdir(parents=True, exist_ok=True)
cfg = {
    "mode": MODE, "input_root": INPUT_ROOT, "working": WORKING,
    "scratch": LOCAL + "/scratch", "data_root": LOCAL + "/data", "hf_home": LOCAL + "/hf",
    "venv": VENV, "device": "cuda", "num_workers": NUM_WORKERS, "seed": SEED,
    "bench_batch_sizes": BENCH_BATCH_SIZES, "bench_amp": BENCH_AMP,
    "bench_steps": BENCH_STEPS, "bench_warmup_steps": BENCH_WARMUP_STEPS,
    "train_steps": TRAIN_STEPS, "train_batch_size": TRAIN_BATCH_SIZE, "train_amp": TRAIN_AMP,
    "save_freq": SAVE_FREQ, "log_freq": 50, "session_hours": SESSION_HOURS,
    "video_backend": VIDEO_BACKEND, "fp32_weights": FP32_WEIGHTS,
    "keep_every": KEEP_EVERY, "freeze": FREEZE,
}
Path(WORKING, "config.json").write_text(json.dumps(cfg, indent=2))
p = subprocess.Popen([sys.executable, "-u", "/tmp/smolvla_kaggle_driver.py", f"{WORKING}/config.json"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, f"driver exited with {p.returncode}; see {WORKING}/logs/"

In [ ]:
# ---- 5. What this session produced ---------------------------------------------
import subprocess
print(subprocess.run(f"cd {WORKING} && du -sh * 2>/dev/null; ls -l run/checkpoints 2>/dev/null; "
                     "cat train_summary.json benchmark.json 2>/dev/null", shell=True,
                     capture_output=True, text=True).stdout)